# 2. Train and compare AR vs NCM using a saved Kaggle Dataset

Enable **GPU** and **Internet**, and attach the private Dataset produced by notebook 1.
This notebook does not download TinyStories. It reads the saved data from
`/kaggle/input/` and saves experiments under `/kaggle/working/ncm-experiment/<run-folder>/`.

Set `PROFILE = 'paper'` in the configuration cell to select:

- Maximum 50,000 updates and global batch 2,048; optional validation early stopping.
- Compiled training; `PRECISION='fp16'` for T4 or `'bf16'` on supported GPUs.
- Published optimizer settings; official NCM sampling, 5 seeds × 256 samples,
  generation batch 128, GPT-2-large scoring.
- Independent output/checkpoint folders, with no automatic reuse of pilot weights.

The default `pilot` remains a 100-update implementation check. Paper mode retains the full published training budget. FP16 is an explicit
precision adaptation, recorded in metadata. This is **not verified equivalence of unpublished training details**.
The configuration prints and saves all reconstructed choices (LR curve, corruption
endpoints, loss exponent, AR weight tying, padding and data sampling policies).
The official pretrained VQ tokenizer and existing code cache are reused.

Set `TRAIN_MODELS = ['ncm']` to spend the entire session training NCM without starting AR. Keep the same profile and precision as any checkpoint you resume.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/kuchidareo/3rd-contribution.git'  # Set the 3rd-contribution repository URL.
REPO_REF = 'main'  # Prefer a full commit containing the baseline implementation.
CHECKOUT = Path('/kaggle/working/3rd-contribution')
REPO = CHECKOUT / '261119-NCM-replicating/ncm'

def run(command, cwd=REPO):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if not CHECKOUT.exists():
    run(['git', 'clone', REPO_URL, CHECKOUT], cwd='/kaggle/working')
run(['git', 'fetch', 'origin', REPO_REF], cwd=CHECKOUT)
run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=CHECKOUT)
required = [REPO / 'baseline_data/prepare.py', REPO / 'training/train_ar.py',
            REPO / 'config/baseline/tinystories.yaml', REPO / 'training/profiles.py']
if any(not p.is_file() for p in required):
    raise RuntimeError('This revision lacks the baseline code. Commit/push the local changes, then set REPO_REF.')
os.chdir(REPO)
# Keep all downloads and environment records inside the preparation/training output.
run(['bash', 'scripts/kaggle_setup.sh'])


In [ ]:
# Fail before any dataset download if the model/training regression tests fail.
run([sys.executable, '-m', 'pytest', 'tests', '-q'])


In [ ]:
import json
import torch
from omegaconf import OmegaConf

# EDIT: folder containing prepared/, codes/ and artifacts/ in your attached Dataset.
DATASET_ROOT = Path('/kaggle/input/ncm-tinystories/ncm-tinystories')
WORK_BASE = Path('/kaggle/working/ncm-experiment')

TRAIN_MODELS = ['ar', 'ncm']  # Set ['ncm'] to train ONLY NCM; ['ar'] for AR only.
PROFILE = 'pilot'  # Set to 'paper' for 50,000 updates and global batch 2,048.
PRECISION = 'fp16'  # T4: fp16. Native BF16-capable GPU: use bf16 to match paper precision.
PILOT_STEPS = 100
PILOT_GLOBAL_BATCH = 32
MICRO_BATCH = 2
VALIDATE_EVERY = 100  # Use 200 if validation overhead is too high.
CHECKPOINT_EVERY = 100
EARLY_STOPPING = True
EARLY_STOP_PATIENCE = 5  # Consecutive validation checks without sufficient improvement.
EARLY_STOP_MIN_DELTA = 0.001  # Absolute validation-loss improvement.
EARLY_STOP_MIN_STEPS = 500  # Never stops before the configured warmup ends.
EVALUATION_CHECKPOINT = 'best'  # 'best' selects lowest clean validation loss; 'latest' is also supported.
# Optional read-only checkpoints from a previous saved Kaggle output.
AR_RESUME = None  # e.g. '/kaggle/input/previous-run/ncm-experiment/paper/runs/ar/latest.pt'
NCM_RESUME = None
# To end this session at an absolute update while keeping the planned LR horizon:
STOP_AT_UPDATE = None
REBUILD_CODES = False  # True only if the attached code cache is absent/incompatible.
NCM_MODEL_OVERRIDES = {}  # Future generator config changes do not require new text data.

if not TRAIN_MODELS or len(set(TRAIN_MODELS)) != len(TRAIN_MODELS) or set(TRAIN_MODELS) - {'ar', 'ncm'}:
    raise ValueError('TRAIN_MODELS must contain ar and/or ncm, without duplicates.')

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before selecting a training profile.')
if not (DATASET_ROOT / 'prepared/manifest.json').is_file():
    raise FileNotFoundError('Set DATASET_ROOT to the attached folder containing prepared/manifest.json.')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
# Reload after Git checkout: notebook kernels otherwise keep old imported functions.
import importlib
import inspect
importlib.invalidate_caches()
import training.profiles as profile_module
if Path(profile_module.__file__).resolve() != (REPO / 'training/profiles.py').resolve():
    raise RuntimeError('training.profiles was imported from another checkout. Restart the kernel and rerun setup.')
profile_module = importlib.reload(profile_module)
configure_profile = profile_module.configure_profile
check_checkpoint_progress = profile_module.check_checkpoint_progress
print('Profile module:', profile_module.__file__)
print('Profile signature:', inspect.signature(configure_profile))
if 'precision' not in inspect.signature(configure_profile).parameters:
    raise RuntimeError('The checked-out training/profiles.py predates FP16 support. Push the updated repository and rerun the clone/checkout cell.')

cfg = OmegaConf.load(REPO / 'config/baseline/tinystories.yaml')
official = OmegaConf.load(REPO / cfg.official_config)
cfg = OmegaConf.create(configure_profile(
    OmegaConf.to_container(cfg, resolve=True),
    OmegaConf.to_container(official, resolve=True),
    PROFILE,
    bf16_supported=(torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported()),
    micro_batch=MICRO_BATCH,
    pilot_steps=PILOT_STEPS,
    pilot_global_batch=PILOT_GLOBAL_BATCH,
    model_overrides=NCM_MODEL_OVERRIDES,
    precision=PRECISION,
))
cfg.training.validation_every = VALIDATE_EVERY
cfg.training.checkpoint_every = CHECKPOINT_EVERY
cfg.training.early_stopping = dict(enabled=EARLY_STOPPING, patience=EARLY_STOP_PATIENCE,
                                  min_delta=EARLY_STOP_MIN_DELTA, min_steps=EARLY_STOP_MIN_STEPS)
if EVALUATION_CHECKPOINT not in {'best', 'latest'}:
    raise ValueError('EVALUATION_CHECKPOINT must be best or latest')
cfg.experiment.completion_required_for_final_comparison = not EARLY_STOPPING and PROFILE == 'paper'
cfg.experiment.early_stopping_is_paper_deviation = EARLY_STOPPING and PROFILE == 'paper'
# FP16 paper runs are separate from BF16 paper and pilot checkpoints.
RUN_FOLDER = 'paper-fp16' if PROFILE == 'paper' and cfg.precision == 'fp16' else PROFILE
if cfg.get('ncm_selective_fp32', False):
    RUN_FOLDER += '-selective'  # Keep the changed NCM numerical policy separate.
WORK = WORK_BASE / RUN_FOLDER
cfg.paths.artifacts = str(DATASET_ROOT / 'artifacts/tinystories')
cfg.paths.tokenizer_checkpoint = str(DATASET_ROOT / 'artifacts/tinystories/tokenizer.pt')
cfg.paths.data = str(DATASET_ROOT / 'prepared')
cfg.paths.codes = str(WORK / 'codes') if REBUILD_CODES else str(DATASET_ROOT / 'codes')
if STOP_AT_UPDATE is not None and not 1 <= STOP_AT_UPDATE <= cfg.training.generator.steps:
    raise ValueError('STOP_AT_UPDATE must be between 1 and the planned total update count.')
WORK.mkdir(parents=True, exist_ok=True)
CONFIG = WORK / 'experiment.yaml'
OmegaConf.save(cfg, CONFIG)

print('Profile:', PROFILE)
print('GPU:', torch.cuda.get_device_name())
print('Output:', WORK)
print('Updates per generator:', cfg.training.generator.steps)
print('Global batch:', cfg.training.generator.global_batch_size)
print('Gradient accumulation:', cfg.experiment.gradient_accumulation_steps)
print('Planned text positions per generator:', cfg.experiment.target_generator_text_positions)
print('Quality samples:', len(cfg.evaluation.seeds), 'seeds x', cfg.evaluation.samples_per_seed)
print('Generation batch:', cfg.evaluation.batch_size)
print('Precision:', cfg.precision, '| compile training:', cfg.training.compile)
print('NCM selective FP32:', cfg.ncm_selective_fp32)
print('NCM policy:', cfg.experiment.ncm_precision_policy)
print('Reconstructed training choices (NOT confirmed author settings):')
print(OmegaConf.to_yaml(cfg.experiment.training_assumptions))
print('Hardware deviations:', list(cfg.experiment.hardware_deviations))
print('Validation/checkpoint intervals:', VALIDATE_EVERY, CHECKPOINT_EVERY)
print('Early stopping:', OmegaConf.to_yaml(cfg.training.early_stopping))
print('Paper steps are a maximum when early stopping is enabled; selected checkpoints may have different training budgets.')


In [ ]:
# Read-only cache reuse: no dataset download or copy is needed.
if REBUILD_CODES:
    if not (Path(cfg.paths.codes) / 'manifest.json').exists():
        run([sys.executable, '-m', 'baseline_data.encode_codes', '--config', CONFIG,
             '--batch-size', '16'])
elif not (Path(cfg.paths.codes) / 'manifest.json').is_file():
    raise FileNotFoundError('Attach cached codes or set REBUILD_CODES=True and rerun the configuration cell.')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from training.runtime import load_config
from training.train import verify_data
verify_data(load_config(CONFIG), with_codes=True)
print('Attached dataset, BPE and frozen-tokenizer cache verified.')


In [ ]:
def train_model(kind, resume=None):
    out = WORK / 'runs' / kind
    command = [sys.executable, '-m', f'training.train_{kind}', '--config', CONFIG, '--out', out]
    checkpoint = out / 'latest.pt'
    source = checkpoint if checkpoint.exists() else (Path(resume) if resume else None)
    completed = 0
    if source is not None:
        # CPU mmap reads metadata without bringing optimizer tensors onto the GPU.
        header = torch.load(source, map_location='cpu', weights_only=False, mmap=True)
        completed = check_checkpoint_progress(kind, header, PROFILE, cfg.training.generator.steps, precision=cfg.precision)
        del header
        command += ['--resume', source]
    stop = STOP_AT_UPDATE if STOP_AT_UPDATE is not None else cfg.training.generator.steps
    if stop < completed:
        raise ValueError(f'{kind}: checkpoint already has {completed} updates; session stop is {stop}.')
    if STOP_AT_UPDATE is not None:
        command += ['--max-steps', str(STOP_AT_UPDATE)]
    print(f'{kind}: completed {completed}; session target {stop}; full target {cfg.training.generator.steps}')
    run(command)

if 'ar' in TRAIN_MODELS:
    train_model('ar', AR_RESUME)
else:
    print('AR training skipped by TRAIN_MODELS.')


In [ ]:
if 'ncm' in TRAIN_MODELS:
    train_model('ncm', NCM_RESUME)
else:
    print('NCM training skipped by TRAIN_MODELS.')


## Evaluate separately from training

These cells generate samples and benchmark each model in a separate process.
Perplexity scoring also runs separately to release generator GPU memory first.
For a new measurement, use a new result directory instead of overwriting an existing
run. The pilot's scores establish a working pipeline, not reproduction of final paper
results. Training-loss scales differ between AR text tokens and NCM latent codes.

For **one-model training**, stop here and use notebook **03_evaluate_tinystories_kaggle.ipynb** to evaluate the saved checkpoint. The comparison cells below require both models. Notebook 3 also accepts checkpoints from a session interrupted by the time limit.


In [ ]:
# Accept a completed budget or explicit validation early stopping. A session
# interrupted by STOP_AT_UPDATE still requires a resume before final evaluation.
RESULT_TAG = 'comparison-best-01'  # Change for a fresh measurement.
RESULTS = WORK / 'results' / RESULT_TAG
ALLOW_UNMATCHED_COMPARISON = False
selected_checkpoints = {}
for kind in ['ar', 'ncm']:
    latest = WORK / 'runs' / kind / 'latest.pt'
    header = torch.load(latest, map_location='cpu', weights_only=False, mmap=True)
    completed = check_checkpoint_progress(kind, header, PROFILE, cfg.training.generator.steps, precision=cfg.precision)
    early_stopped = header.get('stop_reason') == 'early_stopping'
    if PROFILE == 'paper' and completed != cfg.training.generator.steps:
        if not (cfg.training.early_stopping.enabled and early_stopped):
            raise RuntimeError(f'{kind}: {completed}/{cfg.training.generator.steps} updates completed without early stopping. Save outputs and resume, or wait for validation early stopping.')
    del header
    selected = WORK / 'runs' / kind / f'{EVALUATION_CHECKPOINT}.pt'
    header = torch.load(selected, map_location='cpu', weights_only=False, mmap=True)
    selected_step = check_checkpoint_progress(kind, header, PROFILE, cfg.training.generator.steps, precision=cfg.precision)
    ALLOW_UNMATCHED_COMPARISON |= selected_step < cfg.training.generator.steps
    selected_checkpoints[kind] = selected
    print(f'{kind}: evaluating {EVALUATION_CHECKPOINT}.pt at update {selected_step}; latest update {completed}')
    del header
if ALLOW_UNMATCHED_COMPARISON:
    print('Validation-selected partial-training comparison: not a completed paper reproduction or a matched-budget claim.')
for kind in ['ar', 'ncm']:
    run([sys.executable, '-m', 'evaluation.evaluate', '--config', CONFIG,
         '--model', kind, '--checkpoint', selected_checkpoints[kind], '--out', RESULTS / kind])


In [ ]:
for kind in ['ar', 'ncm']:
    run([sys.executable, '-m', 'evaluation.score', '--run', RESULTS / kind,
         '--batch-size', str(cfg.evaluation.scoring_batch_size)])
compare_command = [sys.executable, '-m', 'evaluation.compare', '--ar', RESULTS / 'ar',
                   '--ncm', RESULTS / 'ncm', '--out', RESULTS / 'comparison']
if ALLOW_UNMATCHED_COMPARISON:
    compare_command += ['--allow-unmatched']
run(compare_command)


In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(RESULTS / 'comparison/comparison.csv'))
display(Image(filename=str(RESULTS / 'comparison/comparison.png')))
for kind in ['ar', 'ncm']:
    print(f'\n{kind.upper()} generated samples:')
    rows = (RESULTS / kind / 'samples.jsonl').read_text().splitlines()
    for row in rows[:3]:
        print(json.loads(row)['text'], '\n')


## Save and resume

For paper FP16 preserve `ncm-experiment/paper-fp16/`; for BF16 preserve
`ncm-experiment/paper/`, including `experiment.yaml`,
`runs/*/latest.pt` **and** `runs/*/best.pt`, metrics and results. Attach that saved output next session and
set `AR_RESUME` / `NCM_RESUME` to the matching **paper** checkpoints with the same precision. Pilot mode uses
`ncm-experiment/pilot/`. Earlier notebook versions used `ncm-experiment/runs/`; those
pilot checkpoints cannot be resumed under the paper training recipe.

`STOP_AT_UPDATE` ends a session at an absolute update without shortening the planned
50,000-update learning-rate horizon. Evaluation accepts **both** completed training runs or runs stopped explicitly by validation early stopping. It selects `best.pt` by default and labels shorter runs as partial-training comparisons. Save outputs and resume instead of treating a
partial run as the final result. With `MICRO_BATCH=2`, paper mode accumulates 1,024
micro batches per optimizer update; full training can require many sessions.

On T4 set `PRECISION='fp16'`; on supported GPUs set `'bf16'` to match paper
precision. There is no silent precision fallback. FP16 overflows lower the loss
scale and retry the same accumulated batch without advancing optimizer updates.
Retries and attempted token exposure are logged separately. Repeated failures
stop at a bounded retry limit, preserving the checkpoint. Generation batch 128 is also explicit: reduce it only for a
separately labelled experiment if memory is insufficient.

The original release does not provide a complete training recipe or AR implementation.
Thus paper mode retains documented reconstruction choices; it does not guarantee the
published PPL. For architecture variants, reuse the prepared data and code cache with
a separate experiment configuration rather than modifying the strict paper profile.

FP16 selective-precision runs now use `paper-fp16-selective/` (pilot: `pilot-selective/`). The Transformer remains FP16 mixed precision; frozen VQ conditioning and tied codebook heads use FP32. This is an explicit adaptation, not the paper BF16 policy. Do not resume an older NCM FP16 checkpoint under the new policy; signatures reject that change. Existing AR checkpoints remain available for evaluation in notebook 3.
